# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/talhahmad-webdev/Machine-Learning-Engineering-internship--FlyrankAI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

One row represents the daily performance of one content page for one pseudonymized client.

## Time Window

For this assignment, I will use **March 2026 (2026-03)** as the analysis window because it is a mid-panel month. According to the FlyRank data guidelines, the final month (June 2026) should be treated as a sealed test period and should not be used for feature development.

In [1]:
from google.colab import userdata
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_token")

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=HF_TOKEN
)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [2]:
from itertools import islice
import pandas as pd

# Take a sample of rows
sample = list(islice(ds, 1000))
df = pd.DataFrame(sample)

# Verify the grain (Unit of Analysis)
duplicates = df.duplicated(
    subset=["report_date", "client_hash_id", "content_hash_id"]
).sum()

print("=== Unit of Analysis Verification ===")
print(f"Sample Rows: {len(df)}")
print(f"Duplicate combinations: {duplicates}")

if duplicates == 0:
    print("✔ The sampled data supports one row per report_date + client + content.")

# Verify available date range in the sample
print("\n=== Sample Time Window ===")
print("Earliest Date:", df["report_date"].min())
print("Latest Date:", df["report_date"].max())

print("\nNote:")
print("This sample starts at the beginning of the dataset. The project analysis will use March 2026 (2026-03), following the FlyRank assignment guidelines.")

=== Unit of Analysis Verification ===
Sample Rows: 1000
Duplicate combinations: 0
✔ The sampled data supports one row per report_date + client + content.

=== Sample Time Window ===
Earliest Date: 2025-01-27
Latest Date: 2025-01-30

Note:
This sample starts at the beginning of the dataset. The project analysis will use March 2026 (2026-03), following the FlyRank assignment guidelines.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features

The following fields will be used as input features because they are available before making a content refresh decision:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- sessions_organic

---

## Label / Proxy

The output is a **Content Opportunity Score**, which ranks content pages based on their priority for refresh.

---

## Context

These fields identify each record and are used for grouping or analysis, not for model training:

- report_date
- client_hash_id
- content_hash_id

---

## Excluded

The following fields are excluded:

- Future information (not available at decision time)
- Label-derived columns (to prevent data leakage)
- Private identifiers as model features

These fields are excluded to ensure honest modeling and avoid data leakage.

In [3]:
# Fields selected for this project

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "sessions_organic"
]

context = [
    "report_date",
    "client_hash_id",
    "content_hash_id"
]

print("=== Feature Columns ===")
for col in features:
    print(f"✔ {col}:", col in df.columns)

print("\n=== Context Columns ===")
for col in context:
    print(f"✔ {col}:", col in df.columns)

=== Feature Columns ===
✔ gsc_impressions: True
✔ gsc_clicks: True
✔ gsc_avg_position: True
✔ ga4_pageviews: True
✔ sessions_organic: True

=== Context Columns ===
✔ report_date: True
✔ client_hash_id: True
✔ content_hash_id: True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
!pip install duckdb -q

In [7]:
import duckdb

con = duckdb.connect()

In [11]:
rel = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 5;
"""

march_df = con.sql(query).df()

march_df

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## Query 1: Verify the Data Grain

The purpose of this query is to verify that each row represents one unique combination of:

- report_date
- client_hash_id
- content_hash_id

If this query returns no rows, then the chosen unit of analysis is valid.

In [12]:
query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_count
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
"""

grain_check = con.sql(query).df()
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count


### Observation

The query returned no duplicate rows. This confirms that each row uniquely represents the daily performance of one content item for one client on one report date.

## Query 2: Verify Row Count and Time Window

This query verifies the number of rows in the selected month and confirms the beginning and ending dates of the analysis window.

In [13]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet');
"""

summary = con.sql(query).df()
summary

,total_rows,earliest_date,latest_date
0,9841378,2026-03-01,2026-03-31


### Observation

The results confirm that the dataset contains records for March 2026 only, matching the selected analysis window.

## Query 3: Verify Data Availability

This query checks how many rows have Google Search Console data available by filtering records where `gsc_data_available IS TRUE`.

In [14]:
query = f"""
SELECT
    COUNT(*) AS rows_with_gsc
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE;
"""

availability = con.sql(query).df()
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_gsc
0,3611061


### Observation

Only rows with `gsc_data_available = TRUE` are considered available for analysis. This helps ensure that feature calculations are based on valid Search Console data.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


This dataset has several important limitations:

1. The history is unbalanced because different clients have different amounts of historical data.
2. Some rows have Google Analytics (GA4) data unavailable, so engagement-related features may be missing.
3. This analysis uses only the March 2026 partition and does not represent the complete dataset.
4. The data is pseudonymized, so no real client or content identities are available.
5. The model should be used to support human decisions rather than make automatic decisions.

In [15]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) AS gsc_available,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) AS ga4_available
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet');
"""

limits = con.sql(query).df()
limits

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,ga4_available
0,9841378,3611061,413966


### Observation

The results show that not every row contains both GSC and GA4 data. This confirms that data availability differs across records and should be considered when building features or interpreting model results.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.